# Volatility Dynamics 03b — calibrating the Module 03 precursor scan

Claude review (**VD 010**). Module 03 scanned every numeric field × 5 pre-event lags, kept each field's best lag and ranked by a discovery score with **no reference distribution**.

03b reruns the identical scan (reproduced exactly; verified against Module 03's own code on synthetic data) on **day-relabel placebo events**: each real q99 onset keeps its asset and exact UTC slot, but its UTC day is permuted. Same-day onsets move together, so event count, clock mix, cross-asset synchronicity and within-day clustering are preserved.

Reported per field: observed score, placebo mean / q95, per-field $p$ and **family-wise** $p$ (vs. the maximum score over all fields in each placebo run). Also for the pre-registered fixed summary (mean percentile over the five lags; no best-lag pick).

**Interpretation boundary.** Placebos fall on ordinary moments, real onsets on volatile ones, so volatility-type fields *should* beat this null. 03b answers "is the ranking beyond selection noise?", not "is anything beyond volatility?" (that is 04b).

In [1]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 300)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_q99_precursors as qp
from data_engineering.modeling import volatility_vd_b_core as vb
from data_engineering.modeling import volatility_q99_precursor_null as pn
START = os.getenv('VD3B_START', '').strip() or None
END = os.getenv('VD3B_END', '').strip() or None
ASSETS = [x.strip().upper() for x in os.getenv('VD3B_ASSETS', '').split(',') if x.strip()]
N_PERM = int(os.getenv('VD3B_N_PERM', str(pn.DEFAULT_N_PERM)))
SEED = int(os.getenv('VD3B_SEED', str(pn.DEFAULT_SEED)))
OUT = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '03b'
OUT.mkdir(parents=True, exist_ok=True)
print('core', vb.VERSION, vb.selftest())
print('03b', pn.VERSION, pn.selftest())


core 0.1.0 {'dense_grid': True, 'us_timestamp_backing': True, 'causal_window_mean': True, 'q99_onsets': True, 'day_relabel_placebo': True, 'no_self_control': True}
03b 0.1.0 {'placebo_null': True, 'vol_proxy_beats_placebo': True, 'noise_not_flagged': True}


In [2]:
def qident(name):
    return '"' + str(name).replace('"', '""') + '"'

def load_frame(columns, start=None, end=None, assets=None):
    cols = list(dict.fromkeys(['state_time', 'underlying', 'return_5m'] + list(columns)))
    where = []
    if start:
        ts = pd.Timestamp(start)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time >= TIMESTAMPTZ '{ts.isoformat()}'")
    if end:
        ts = pd.Timestamp(end)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time < TIMESTAMPTZ '{(ts + pd.Timedelta(days=1)).isoformat()}'")
    if assets:
        where.append('underlying IN (' + ','.join(repr(x) for x in assets) + ')')
    sql = ('SELECT ' + ', '.join(qident(c) for c in cols) + ' FROM time_series_feature_state'
           + ('' if not where else ' WHERE ' + ' AND '.join(where)) + ' ORDER BY underlying, state_time')
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        df = con.execute(sql).df()
    finally:
        con.close()
    if df.empty:
        raise RuntimeError('No rows found.')
    df['state_time'] = pd.to_datetime(df['state_time'], utc=True)
    if df.duplicated(['underlying', 'state_time']).any():
        raise RuntimeError('Duplicate asset/timestamp rows.')
    print(f'rows {len(df):,} | assets {sorted(df.underlying.unique())} | {df.state_time.min()} -> {df.state_time.max()}')
    return df.reset_index(drop=True)

def schema_columns():
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        return con.execute('DESCRIBE time_series_feature_state').df()
    finally:
        con.close()

schema = schema_columns()
predictors, inventory = qp.eligible_predictor_columns(schema)
print('eligible predictors (Module 03 rule):', len(predictors))
df = load_frame(predictors, START, END, ASSETS)


eligible predictors (Module 03 rule): 205
rows 42,336 | assets ['AVAX', 'BTC', 'ETH', 'HYPE', 'SOL', 'TRX', 'XRP'] | 2026-09-19 08:00:00+00:00 -> 2026-10-10 07:55:00+00:00


In [3]:
grid = vb.build_grid(df)
thr = vb.q99_thresholds(grid)                    # full-sample, as Module 03
tail = vb.tail_flags(grid, thr)
event_rows = np.flatnonzero(vb.onset_flags(grid, tail))
eligible = vb.eligible_origins(grid, tail)
X = grid.feature_matrix(predictors)
print('asset-event onsets:', len(event_rows), '| unique timestamps:', len(np.unique(grid.time_ns[event_rows])))
print('valid days for relabelling:', len(vb.valid_days(grid)))


asset-event onsets: 337 | unique timestamps: 198
valid days for relabelling: 19


In [4]:
observed = pn.scan(grid, X, event_rows, eligible)
observed.insert(0, 'feature', [predictors[i] for i in observed.feature_index])
ref = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '03' / '03_precursor_rankings.csv'
if ref.exists():
    r3 = pd.read_csv(ref)[['feature', 'discovery_score']].merge(observed[['feature', 'discovery_score']], on='feature', suffixes=('_module03', '_03b'))
    diff = (r3.discovery_score_module03 - r3.discovery_score_03b).abs()
    print(f'reproduction vs Module 03 output: {len(r3)} features, max |diff| = {diff.max():.2e} (non-zero only if the database changed since Module 03 ran)')
display(observed.sort_values('discovery_score', ascending=False).head(30).round(4))


reproduction vs Module 03 output: 197 features, max |diff| = 7.96e-02 (non-zero only if the database changed since Module 03 ran)


,feature,feature_index,window_mean_percentile,window_n_unique_event_times,discovery_score,best_rel_bar,best_n_unique_event_times,best_mean_percentile,n_assets,asset_direction_consistency,lag_direction_consistency
79,mid_path_max_abs_log_return,79,0.6785,29,0.2551,-3.0,29.0,0.7551,7.0,1.0,1.0
75,mid_path_realized_variance,75,0.6963,29,0.2519,-1.0,29.0,0.7519,7.0,1.0,1.0
65,book_obi_20_path_roughness_per_min,65,0.6887,56,0.2258,-1.0,56.0,0.7258,7.0,1.0,1.0
91,intrabar_range_bps,91,0.6696,171,0.2159,-1.0,158.0,0.7159,7.0,1.0,1.0
69,book_obi_20_raw_zero_cross_count,69,0.6818,56,0.2113,-6.0,55.0,0.7113,7.0,1.0,1.0
144,iv_put_25d,144,0.6780,181,0.2100,-3.0,165.0,0.7100,7.0,1.0,1.0
70,book_obi_20_raw_zero_cross_rate,70,0.6630,56,0.2016,-1.0,56.0,0.7016,7.0,1.0,1.0
106,realized_vol_1h_annualized,106,0.6772,189,0.1982,-1.0,168.0,0.6982,7.0,1.0,1.0
105,realized_vol_1h,105,0.6772,189,0.1982,-1.0,168.0,0.6982,7.0,1.0,1.0
108,realized_vol_4h_annualized,108,0.6882,97,0.1954,-2.0,83.0,0.6954,7.0,1.0,1.0


In [5]:
t0 = time.time()
S, W, counts = pn.placebo_null(grid, X, event_rows, eligible, n_perm=N_PERM, seed=SEED, progress_every=50)
print(f'{N_PERM} placebo scans in {time.time()-t0:.0f}s')
display(counts.describe().round(1))
calibrated = pn.calibrate(observed, predictors, S, W)
fw = pn.familywise_thresholds(S, W)
display(fw.round(4))
display(calibrated.head(40).round(4))
print('fields with family-wise p < 0.05 (discovery score):', int((calibrated.p_familywise_score < 0.05).sum()))


placebo 50/500
placebo 100/500
placebo 150/500
placebo 200/500
placebo 250/500
placebo 300/500
placebo 350/500
placebo 400/500
placebo 450/500
placebo 500/500
500 placebo scans in 32s


,perm,n_placebo_asset_events,n_placebo_unique_times
count,500.0,500.0,500.0
mean,249.5,288.6,170.5
std,144.5,22.1,12.4
min,0.0,197.0,116.0
25%,124.8,276.0,163.0
50%,249.5,293.5,172.0
75%,374.2,305.2,180.0
max,499.0,325.0,190.0


,statistic,null_q50,null_q95,null_q99
0,max_discovery_score,0.1186,0.1927,0.2316
1,max_abs_window_dev,0.0968,0.1569,0.1893


,feature,observed_discovery_score,best_rel_min,n_unique_event_times_best,observed_window_mean_percentile,window_n_unique_event_times,null_n_valid,null_mean_score,null_q95_score,p_feature_score,p_familywise_score,null_q95_abs_window_dev,p_feature_window,p_familywise_window
0,mid_path_max_abs_log_return,0.2551,-15.0,29.0,0.6785,29,497,0.0585,0.1469,0.0080,0.0100,0.0911,0.0060,0.0220
1,mid_path_realized_variance,0.2519,-5.0,29.0,0.6963,29,497,0.0654,0.1587,0.0080,0.0100,0.1163,0.0060,0.0080
2,book_obi_20_path_roughness_per_min,0.2258,-5.0,56.0,0.6887,56,500,0.0551,0.1511,0.0040,0.0160,0.1243,0.0040,0.0140
3,intrabar_range_bps,0.2159,-5.0,158.0,0.6696,171,500,0.0356,0.0864,0.0020,0.0240,0.0633,0.0020,0.0279
4,book_obi_20_raw_zero_cross_count,0.2113,-30.0,55.0,0.6818,56,500,0.0525,0.1346,0.0040,0.0299,0.1022,0.0020,0.0180
5,iv_put_25d,0.2100,-15.0,165.0,0.6780,181,500,0.0452,0.1174,0.0020,0.0299,0.1039,0.0040,0.0220
6,book_obi_20_raw_zero_cross_rate,0.2016,-5.0,56.0,0.6630,56,500,0.0456,0.1178,0.0020,0.0439,0.0852,0.0020,0.0339
7,realized_vol_1h,0.1982,-5.0,168.0,0.6772,189,500,0.0370,0.0892,0.0020,0.0479,0.0770,0.0020,0.0220
8,realized_vol_1h_annualized,0.1982,-5.0,168.0,0.6772,189,500,0.0370,0.0892,0.0020,0.0479,0.0770,0.0020,0.0220
9,realized_vol_4h,0.1954,-10.0,83.0,0.6882,97,500,0.0562,0.1374,0.0040,0.0519,0.1301,0.0020,0.0140


fields with family-wise p < 0.05 (discovery score): 9


In [6]:
inventory.to_csv(OUT / '03b_feature_inventory.csv', index=False)
pd.DataFrame([{'underlying': k, 'q99_threshold': v} for k, v in thr.items()]).to_csv(OUT / '03b_q99_thresholds.csv', index=False)
observed.to_csv(OUT / '03b_observed_rankings.csv', index=False)
calibrated.to_csv(OUT / '03b_calibrated_rankings.csv', index=False)
fw.to_csv(OUT / '03b_familywise_null.csv', index=False)
counts.to_csv(OUT / '03b_placebo_counts.csv', index=False)
meta = {
    'version': pn.VERSION, 'core_version': vb.VERSION, 'module': '03b_q99_precursor_null',
    'scan': 'Module 03 clock-only controls, rank lags -60,-30,-15,-10,-5 min, best-lag discovery score (reproduced exactly)',
    'null': 'day-relabel placebo: UTC day permuted, asset and exact slot kept; same-day events move together',
    'n_perm': N_PERM, 'seed': SEED, 'n_predictors': len(predictors),
    'n_asset_event_onsets': int(len(event_rows)), 'n_unique_event_times': int(len(np.unique(grid.time_ns[event_rows]))),
    'start': START or 'earliest_available', 'end': END or 'latest_available',
    'first_time': str(df.state_time.min()), 'last_time': str(df.state_time.max()), 'n_rows': int(len(df)),
}
(OUT / '03b_run_metadata.json').write_text(json.dumps(meta, indent=2))
print('wrote', OUT)


wrote D:\Projects\deribit_market_data\notebooks\Volatility_Dynamics\03b
